# Denoising Autoencoders

## Checking GPU Usage

Go to the `Runtime > Change runtime type` menu and check that we're on Python 3 and that the hardware accelerator is set to "GPU".

In [ ]:
!nvidia-smi

## Installing and Importing PyTorch Lightning and the Other Required Libraries

In [ ]:
!pip install -q lightning

In [ ]:
import lightning
import matplotlib.pyplot as plt
import torch
import torchvision
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

## Loading MNIST

We'll use slightly different preprocessing than usual: since we want to be able to predict the input values at the output (the principle of autoencoding), we'll simply project these values into $[0, 1]$ instead of $[0, 255]$. Note that we usually don't do that: we normalize by centering on zero and dividing by the standard deviation.

In [ ]:
train_data = torchvision.datasets.MNIST("data", train=True, download=True)
test_data = torchvision.datasets.MNIST("data", train=False, download=True)
nb_classes = 10
input_dim = 28 * 28
X_train = train_data.data.reshape(-1, input_dim).float() / 255.0
X_test = test_data.data.reshape(-1, input_dim).float() / 255.0
y_test = test_data.targets

## Applying Gaussian Noise

In [ ]:
noise_factor = 0.5
X_train_noisy = X_train + torch.normal(0., noise_factor, X_train.shape)
X_test_noisy = X_test + torch.normal(0., noise_factor, X_test.shape)

# Clip the values to avoid pixels whiter than white (or blacker
# than black)
X_train_noisy.clamp_(0, 1)
X_test_noisy.clamp_(0, 1)

In [ ]:
n = 10
f, ax = plt.subplots(1, n, figsize=(n * 1.4, 2))
for i in range(n):
  ax[i].imshow(X_test_noisy[i].reshape(28, 28), cmap="gray_r")
  ax[i].set_title(int(y_test[i]))
  ax[i].axis("off")
plt.show()

## Creating the Denoising Autoencoder

In [ ]:
class AutoEncoder(lightning.LightningModule):
  """Lightning wrapper: reconstruct the clean image from the noisy one."""

  def __init__(self,
               model: nn.Module,
               loss: nn.Module,
               learning_rate: float = 1e-3,
               input_shape: tuple[int, ...] = (input_dim,)) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model", "loss"])
    self.model = model
    self.loss = loss
    # An example input lets Lightning display the shapes of the input and output
    # tensors of each layer in the model summary
    self.example_input_array = torch.zeros(1, *input_shape)

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.model(images)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor],
            stage: str) -> torch.Tensor:
    noisy_images, images = batch
    loss = self.loss(self(noisy_images), images)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True,
             prog_bar=True)
    return loss

  def training_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                    batch_index: int) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                      batch_index: int) -> torch.Tensor:
    return self._step(batch, "val")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(),
                            lr=self.hparams.learning_rate)


@torch.no_grad()
def predict(model: nn.Module,
            X: torch.Tensor,
            batch_size: int = 256) -> torch.Tensor:
  """Apply a network to every row of X, batch by batch."""
  device = next(model.parameters()).device
  model.eval()
  return torch.cat([model(batch.to(device)).cpu()
                    for batch in X.split(batch_size)])

In [ ]:
# Your code here
encoding_dim = 12

autoencoder = nn.Sequential()

### Solution

In [ ]:
encoding_dim = 12

autoencoder = nn.Sequential(
    nn.Linear(input_dim, encoding_dim),
    nn.ReLU(),
    nn.Linear(encoding_dim, input_dim),
    nn.Sigmoid())

model = AutoEncoder(autoencoder, loss=nn.MSELoss())

print(ModelSummary(model, max_depth=-1))

## Training

*Write the lines that train your autoencoder:*

- *50 epochs should be enough*
- *Use a batch size of 256*

In [ ]:
# Your code here

### Solution

In [ ]:
batch_size = 256

# 20% of the training data is set aside for validation
train_dataset, val_dataset = random_split(
    TensorDataset(X_train_noisy, X_train), [0.8, 0.2])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

trainer = lightning.Trainer(max_epochs=50,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="autoencoder"),
                            enable_checkpointing=False)
trainer.fit(model, train_loader, val_loader)

## Test Set

Autoencode the test images and store the resulting images in the variable `X_test_noisy_pred`

In [ ]:
# Your code here
X_test_noisy_pred = X_test_noisy

### Solution

In [ ]:
X_test_noisy_pred = predict(autoencoder, X_test_noisy)

## Visual Display of the Performance

In [ ]:
n = 10
_, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
for i, (ax_top, ax_bottom) in enumerate(ax.T):
  # The original on top
  ax_top.imshow(X_test_noisy[i].reshape(28, 28), cmap="gray_r")
  ax_top.set_title(str(int(y_test[i])))
  ax_top.axis("off")

  # The reconstruction at the bottom
  ax_bottom.imshow(X_test_noisy_pred[i].reshape(28, 28), cmap="gray_r")
  ax_bottom.axis("off")
plt.show()

## Try With More Neurons

What happens?

In [ ]:
# Your code here

### Solution

In [ ]:
encoding_dim = 64

autoencoder = nn.Sequential(
    nn.Linear(input_dim, encoding_dim),
    nn.ReLU(),
    nn.Linear(encoding_dim, input_dim),
    nn.Sigmoid())
model = AutoEncoder(autoencoder, loss=nn.MSELoss())

trainer = lightning.Trainer(max_epochs=50,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="autoencoder_64"),
                            enable_checkpointing=False)
trainer.fit(model, train_loader, val_loader)

X_test_noisy_pred = predict(autoencoder, X_test_noisy)
n = 10
_, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
for i, (ax_top, ax_bottom) in enumerate(ax.T):
  # The original on top
  ax_top.imshow(X_test_noisy[i].reshape(28, 28), cmap="gray_r")
  ax_top.set_title(str(int(y_test[i])))
  ax_top.axis("off")

  # The reconstruction at the bottom
  ax_bottom.imshow(X_test_noisy_pred[i].reshape(28, 28), cmap="gray_r")
  ax_bottom.axis("off")
plt.show()

With a 64-neuron code instead of 12, the validation loss decreases and the denoised digits are sharper: the bottleneck lets more details through. The network still doesn't just copy its input, noise included: the denoising objective (noisy input, clean target) prevents it. Without it, a wide enough code would allow learning the identity, and the autoencoder would become useless.

## Using Convolutional Networks

To do so, each image must be put back into a 1 × 28 × 28 shape. CNNs need this channel dimension, which PyTorch expects right after the batch dimension (there could be more channels than the gray level: there are 3 for color images, and even more in the intermediate layers of a convolutional network, where the number of input channels of a layer is the number of kernels of the previous layer).

In [ ]:
X_train = X_train.reshape(-1, 1, 28, 28)
X_test = X_test.reshape(-1, 1, 28, 28)
X_train_noisy = X_train_noisy.reshape(-1, 1, 28, 28)
X_test_noisy = X_test_noisy.reshape(-1, 1, 28, 28)

## Creating the Model

We'll use sequences of [`Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) and [`MaxPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) with kernels of 3 × 3 and 2 × 2 respectively for the encoder part. In the convolution, you'll need the `padding="same"` option to avoid border effects (the image being small enough as it is).

For the decoder part, we'll use [`Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) layers of the same kind followed by [`Upsample`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Upsample.html) layers (with a factor of 2), which are the inverse operation of [`MaxPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html).

Feel free to overuse `ModelSummary` to find your way. The goal is to get back a 1 × 28 × 28 image at the output of the last [`Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html). Indeed, ending with an [`Upsample`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Upsample.html) would be a bad idea.

The network will be deep: we'll use ReLU activation functions, except for the last one, where we'll use a sigmoid.

In [ ]:
# Your code here
autoencoder = nn.Sequential()

### Solution

In [ ]:
encoding_dim = 20


def conv(in_channels: int,
         out_channels: int,
         activation: type[nn.Module] = nn.ReLU) -> nn.Sequential:
  return nn.Sequential(
      nn.Conv2d(in_channels, out_channels, (3, 3), padding="same"),
      activation())


def max_pool() -> nn.MaxPool2d:
  return nn.MaxPool2d((2, 2), ceil_mode=True)


def up_sampling() -> nn.Upsample:
  return nn.Upsample(scale_factor=2)


autoencoder = nn.Sequential(
    # Encoding
    conv(1, 32),
    max_pool(),
    conv(32, 32),
    max_pool(),
    conv(32, 1),
    nn.Flatten(),
    nn.Linear(7 * 7, encoding_dim),

    # Decoding
    nn.Linear(encoding_dim, 7 * 7),
    nn.Unflatten(1, (1, 7, 7)),
    conv(1, 32),
    up_sampling(),
    conv(32, 32),
    up_sampling(),
    conv(32, 1, nn.Sigmoid))

model = AutoEncoder(autoencoder,
                    loss=nn.BCELoss(),
                    learning_rate=1e-4,
                    input_shape=X_train.shape[1:])

print(ModelSummary(model, max_depth=-1))

## Training

*Write the lines that train your autoencoder:*

- *100 epochs should be enough*
- *Use a batch size of 256*

In [ ]:
# Your code here

### Solution

In [ ]:
batch_size = 256

train_dataset, val_dataset = random_split(
    TensorDataset(X_train_noisy, X_train), [0.8, 0.2])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

trainer = lightning.Trainer(max_epochs=100,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="conv_autoencoder"),
                            enable_checkpointing=False)
trainer.fit(model, train_loader, val_loader)

## Displaying the Performance

In [ ]:
X_test_noisy_pred = predict(autoencoder, X_test_noisy).reshape(-1, 28, 28)

n = 10

random_indexes = torch.randperm(X_test.shape[0])[:n]

_, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
for (ax_top, ax_bottom), random_index in zip(ax.T, random_indexes):
  # The original image on top
  ax_top.set_title(str(int(y_test[random_index])))
  ax_top.imshow(X_test_noisy[random_index].reshape(28, 28), cmap="gray_r")
  ax_top.axis("off")

  # The reconstructed image at the bottom
  ax_bottom.imshow(X_test_noisy_pred[random_index], cmap="gray_r")
  ax_bottom.axis("off")
plt.show()

## On Non-Noisy Images

In [ ]:
X_test_pred = predict(autoencoder, X_test).reshape(-1, 28, 28)

n = 10

random_indexes = torch.randperm(X_test.shape[0])[:n]

_, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
for (ax_top, ax_bottom), random_index in zip(ax.T, random_indexes):
  # The original image on top
  ax_top.set_title(str(int(y_test[random_index])))
  ax_top.imshow(X_test[random_index].reshape(28, 28), cmap="gray_r")
  ax_top.axis("off")

  # The reconstructed image at the bottom
  ax_bottom.imshow(X_test_pred[random_index], cmap="gray_r")
  ax_bottom.axis("off")
plt.show()